# The Allergen & Food Safety Hazard Compass
## Yelp Dataset — Feasibility Analysis

This notebook checks whether the Yelp dataset contains enough **allergy-related, high-risk reviews** to support the project.

**Goal:** Find reviews where:
1. An allergen keyword appears (allergy, celiac, gluten-free, etc.)
2. The review signals a real safety incident — low star rating OR explicit harm language

In [ ]:
import io, os, subprocess, sys, zipfile
import pandas as pd

DATASET_DIR = r"e:\textminning"
ZIP_PATH    = os.path.join(DATASET_DIR, "yelp-dataset.zip")
JSON_PATH   = os.path.join(DATASET_DIR, "yelp_academic_dataset_review.json")
KAGGLE_DATASET = "yelp-dataset/yelp-dataset"
TARGET_FILE = "yelp_academic_dataset_review.json"

def ensure_download():
    if os.path.exists(ZIP_PATH) or os.path.exists(JSON_PATH):
        print("✅ Dataset found.")
        return
    print("Dataset not found — downloading from Kaggle...")
    try:
        import kaggle
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])
    kaggle_json = os.path.join(os.path.expanduser("~"), ".kaggle", "kaggle.json")
    if not os.path.exists(kaggle_json):
        raise FileNotFoundError(f"Kaggle API key not found at {kaggle_json}")
    subprocess.check_call([
        sys.executable, "-m", "kaggle", "datasets", "download",
        "-d", KAGGLE_DATASET, "-p", DATASET_DIR
    ])
    print("✅ Download complete.")

ensure_download()

## Step 2 — Define Keywords

Two keyword groups:
- **Allergen keywords** — identify reviews that mention any food allergy or hazard
- **Harm keywords** — identify reviews where something actually went wrong (used alongside star rating)

In [ ]:
allergy_keywords = [
    r"\ballerg(?:y|ies|ic)\b",
    r"\bceliac\b",
    r"\bgluten.free\b",
    r"\bcross.contaminat\w*",
    r"\bcontaminat\w*\b",
    r"\banaphylact\w*\b",
    r"\bepi.?pen\b",
    r"\bfood.?poison\w*",
    r"\bgot sick\b",
    r"\bsent me to the hospital\b",
    r"\bwent to the hospital\b",
    r"\bended up in the hospital\b",
]

harm_keywords = [
    r"\bgot sick\b", r"\bfelt sick\b", r"\bstomach.?ache\b",
    r"\ballergic reaction\b", r"\bbroke out\b",
    r"\bsent me to the hospital\b", r"\bended up in the hospital\b",
    r"\bepi.?pen\b", r"\banaphylact\w*\b", r"\bcross.contaminat\w*",
    r"\bfood.?poison\w*",
]

keyword_pattern = "|".join(allergy_keywords)
harm_pattern    = "|".join(harm_keywords)

print(f"Allergen patterns: {len(allergy_keywords)}")
print(f"Harm patterns:     {len(harm_keywords)}")

## Step 3 — Scan the Dataset

In [ ]:
def open_reviews():
    if os.path.exists(JSON_PATH):
        return open(JSON_PATH, "r", encoding="utf-8"), None
    zf = zipfile.ZipFile(ZIP_PATH, "r")
    match = next((n for n in zf.namelist() if TARGET_FILE in n), None)
    if not match:
        raise FileNotFoundError(f"{TARGET_FILE} not found in zip. Contents: {zf.namelist()}")
    return io.TextIOWrapper(zf.open(match), encoding="utf-8"), zf

total_reviews          = 0
total_allergy_reviews  = 0
dangerous_reviews      = 0
all_dangerous          = []

fh, zf = open_reviews()
try:
    for chunk in pd.read_json(fh, lines=True, chunksize=100_000):
        total_reviews += len(chunk)
        text_lower = chunk["text"].str.lower()

        allergy_mask  = text_lower.str.contains(keyword_pattern, na=False, regex=True)
        allergy_chunk = chunk[allergy_mask].copy()
        total_allergy_reviews += len(allergy_chunk)

        harm_mask     = text_lower[allergy_mask].str.contains(harm_pattern, na=False, regex=True)
        dangerous     = allergy_chunk[(allergy_chunk["stars"] <= 2) | harm_mask]
        dangerous_reviews += len(dangerous)
        all_dangerous.append(dangerous)

        print(f"  Scanned {total_reviews:>9,} | allergy: {total_allergy_reviews:,} | high-risk: {dangerous_reviews:,}", end="\r")
finally:
    fh.close()
    if zf: zf.close()

dangerous_df = pd.concat(all_dangerous, ignore_index=True)
print(f"\n✅ Done. {total_reviews:,} reviews scanned.")

## Step 4 — Results Summary

In [ ]:
import matplotlib.pyplot as plt

labels  = ["Total Reviews", "Mention Allergens", "High-Risk"]
counts  = [total_reviews, total_allergy_reviews, dangerous_reviews]
colors  = ["#4c72b0", "#dd8452", "#c44e52"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
bars = axes[0].bar(labels, counts, color=colors, width=0.5)
axes[0].set_title("Review Counts by Category", fontsize=13)
axes[0].set_ylabel("Number of Reviews")
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{val:,}", ha="center", va="bottom", fontsize=10)

# Star distribution of high-risk reviews
star_counts = dangerous_df["stars"].value_counts().sort_index()
axes[1].bar(star_counts.index.astype(str), star_counts.values, color="#c44e52", width=0.5)
axes[1].set_title("Star Rating Distribution — High-Risk Reviews", fontsize=13)
axes[1].set_xlabel("Stars")
axes[1].set_ylabel("Number of Reviews")

plt.tight_layout()
plt.show()

print(f"\n{'='*50}")
print(f"Total reviews scanned:              {total_reviews:>10,}")
print(f"Reviews mentioning allergens:       {total_allergy_reviews:>10,}  ({100*total_allergy_reviews/total_reviews:.2f}%)")
print(f"High-risk reviews:                  {dangerous_reviews:>10,}")
print(f"{'='*50}")
verdict = "✅ SUFFICIENT — Project 1 is viable." if dangerous_reviews >= 2000 else "❌ TOO SPARSE — Consider Project 2."
print(f"\n{verdict}")

## Step 5 — Sample High-Risk Reviews

Reading the actual reviews to confirm they are genuinely about food safety incidents (not false positives).

In [ ]:
sample = dangerous_df[["stars", "text"]].sample(5, random_state=42)

for i, (_, row) in enumerate(sample.iterrows(), 1):
    stars_display = "⭐" * int(row["stars"])
    print(f"--- Sample #{i} | {stars_display} ({int(row['stars'])} stars) ---")
    print(row["text"][:600])
    print()